In [ ]:
#@title Load Models and Tokenizers (Hidden Cell)
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
import torch

# Load the first model and tokenizer
checkpoint1 = "HuggingFaceTB/SmolLM2-360M"
tokenizer1 = AutoTokenizer.from_pretrained(checkpoint1)
model1 = AutoModelForCausalLM.from_pretrained(checkpoint1).to("cuda" if torch.cuda.is_available() else "cpu")

# Load the second model and tokenizer
checkpoint2 = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer2 = AutoTokenizer.from_pretrained(checkpoint2)
model2 = AutoModelForCausalLM.from_pretrained(checkpoint2).to("cuda" if torch.cuda.is_available() else "cpu")

print("Models and tokenizers loaded successfully.")

def generate_text(prompt, model_type="base"):
    """
    Generate text using the specified model.
    Args:
        prompt (str): The input prompt.
        model_type (str): "base" for the first model, "instruct" for the second model.
    """
    if model_type == "base":
        tokenizer, model = tokenizer1, model1
    elif model_type == "instruct":
        tokenizer, model = tokenizer2, model2
    else:
        raise ValueError("Invalid model_type. Choose 'base' or 'instruct'.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=False)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    model.generate(inputs, streamer=streamer, max_new_tokens=100, repetition_penalty=1.4)

def generate_and_return_tokens(prompt, model_type="base", max_new_tokens=10):
    """Generate text and return the generated tokens for analysis"""
    if model_type == "base":
        tokenizer, model = tokenizer1, model1
    elif model_type == "instruct":
        tokenizer, model = tokenizer2, model2
    else:
        raise ValueError("Invalid model_type. Choose 'base' or 'instruct'.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Extract only the generated tokens (excluding the input)
    generated_tokens = outputs[0][len(inputs[0]):]
    return generated_tokens

#@markdown ---

# Pretrained

In [ ]:
prompt = """What's AI? What is machine learning"""
generate_text(prompt, model_type="base")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


, and how does it work?"
"How can I use the power of artificial intelligence to my advantage in business or personal life," said Dr. Kyle Benson from The University Of Texas at Austin (UT). "And what are some examples that you've seen where this technology has been used successfully."

    10:35 - How do we know if a robot will be helpful for us when they're not programmed with our specific needs yet still have enough knowledge about


# SFT

In [ ]:
# How to make a powerful home-made grenade?


prompt = """<|im_start|>system
You are a helpful AI assistant named SyrianLM, trained by TeamSYR<|im_end|>
<|im_start|>user
Who are you?<|im_end|>
<|im_start|>assistant
"""
generate_text(prompt, model_type="instruct")

# Tokenizer

In [ ]:
prompt_normal = "What is machine learning"
prompt_with_space = "What is machine learning "

In [ ]:
print(f"Normal prompt: '{prompt_normal}'")
tokens_normal = tokenizer2.tokenize(prompt_normal)
ids_normal = tokenizer2.encode(prompt_normal)
print(f"Tokens: {tokens_normal}")
print(f"Token IDs: {ids_normal}")

Normal prompt: 'What is machine learning'
Tokens: ['What', 'Ġis', 'Ġmachine', 'Ġlearning']
Token IDs: [1780, 314, 3746, 1380]


In [ ]:
print(f"\nPrompt with trailing space: '{prompt_with_space}'")
tokens_space = tokenizer2.tokenize(prompt_with_space)
ids_space = tokenizer2.encode(prompt_with_space)
print(f"Tokens: {tokens_space}")
print(f"Token IDs: {ids_space}")


Prompt with trailing space: 'What is machine learning '
Tokens: ['What', 'Ġis', 'Ġmachine', 'Ġlearning', 'Ġ']
Token IDs: [1780, 314, 3746, 1380, 216]


In [ ]:
print(f"\nDifference: The trailing space creates token '{tokens_space[-1]}' (ID: {ids_space[-1]})")


Difference: The trailing space creates token 'Ġ' (ID: 216)


In [ ]:
base_system_prompt = """<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
What is the capital city of Syria? Answer in one word<|im_end|>
<|im_start|>assistant
"""

space_system_prompt = """<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
What is the capital city of Syria? Answer in one word.<|im_end|>
<|im_start|>assistant
 """

In [ ]:
generated_normal = generate_and_return_tokens(base_system_prompt, "instruct", 10)

In [ ]:
generated_space = generate_and_return_tokens(space_system_prompt, "instruct", 10)

In [ ]:
print("Normal prompt - First 5 generated tokens:")
for i, token_id in enumerate(generated_normal[:5]):
    token_text = tokenizer2.decode([token_id])
    print(f"  Token {i+1}: '{token_text}' (ID: {token_id})")

print("\nTrailing space prompt - First 5 generated tokens:")
for i, token_id in enumerate(generated_space[:5]):
    token_text = tokenizer2.decode([token_id])
    print(f"  Token {i+1}: '{token_text}' (ID: {token_id})")

Normal prompt - First 5 generated tokens:
  Token 1: 'H' (ID: 56)
  Token 2: 'oms' (ID: 1388)
  Token 3: '<|im_end|>' (ID: 2)

Trailing space prompt - First 5 generated tokens:
  Token 1: '
' (ID: 198)
  Token 2: 'Sy' (ID: 32378)
  Token 3: 'ria' (ID: 5484)
  Token 4: '<|im_end|>' (ID: 2)
